# Some interesting notes
### Resources:
#### https://www.jeremykun.com/2023/08/10/mlir-getting-started/
#### https://github.com/joker-eph/llvm-project-with-mlir/blob/master/mlir/g3doc/WritingAPass.md
#### https://www.stephendiehl.com/posts/mlir_memory/
#### https://mlir.llvm.org/docs/Canonicalization/#canonicalizing-with-rewritepatterns
What is MLIR: In short, it’s a framework for building compilers, with the underlying philosophy that a big compiler should be broken up into lots of small compilers between sub-languages (which compiler folks call “intermediate representations” or “IR”s), where each sub-language is designed to make a particular kind of optimization more natural to express. Hence the MLIR acronym standing for Multi-Level Intermediate Representation.MLIR is relevant for TensorFlow because training and inference can both be thought of as programs whose instructions are things like “2d convolution” and “softmax.” And the process for optimizing those instructions, while converting them to lower level hardware instructions (especially on TPU accelerators) is very much a compilers problem. MLIR breaks the process up into IRs at various levels of abstraction, like Tensor operations, linear algebra, and lower-level control flow.

Two of the central concepts in MLIR are dialects and lowerings. These are the scaffolding within which we can do the truly interesting parts of a compiler—that is, the optimizations and analyses. In traditional compilers, there is typically one “dialect” (called an intermediate representation, or IR) that is the textual or data-structural description of a program within the compiler’s code. For example, in GCC the IR is called GIMPLE, and in LLVM it’s called LLVM-IR. They convert the input program to the IR, do their optimizations, and then convert the optimized IR to machine code.

In MLIR one splits the job into much smaller steps. First, MLIR allows one to define many dialects, some considered “high level” and some “low level,” but each with a set of types, operations, metadata, and semantics that defines what the operations do. Then, one writes a set of lowering passes that incrementally converts different parts of the program from higher level dialects to lower and lower dialects until you get to machine code (or, in many cases, LLVM, which finishes the job). Along the way, optimizing passes are run to make the code more efficient. The main point here is that the high level dialects exist so that they make it easy to write these important optimizing passes. And there’s not a special distinction between lowering passes and optimizing passes, they’re both just called passes in MLIR and are generic IR-rewriting modules.

Aside: From what I can gather, a big part of the motivation for MLIR was to build the affine dialect, which is specifically designed to enable polyhedral optimizations for loop transformations, along with the linalg dialect, which does optimization passes like tiling for low-level ML operations on specialized hardware. Folks built polyhedral optimizations in LLVM and GCC (without affine), and it was a huge pain in the ass, mainly because they had to take a low-level mess of branches and GOTOs and try to reconstruct a simple (‘affine’) for loop structure from it. This was necessary even if the input program was a simple set of for loops, because by the time they got to the compiler, the rigid for loop structure had been discarded. MLIR instead says, keep the structure in the higher level dialect, optimize there, and then discard it when you lower to lower level dialects.

--> MLIR dialects:

-> arith is for low-level arithmetic and boolean conditions on integers and floats. You can define constants, compare integers with arith.cmpi, and do things like add and bit shift (arith.shli is a left shift). 
-> scf, short for “structured control flow,” defines for loops, while loops, and control flow branching. scf.yield defines the “output” value from each region of an if/else operation or loop body which is necessary here because, as you can see, an if operation has a result value.

-> use of EOF in files: Effectively, it creates a file with the content you typed between the <<'EOF' and the ending EOF.

## Check your system's LLVM and MLIR

In [189]:
import os
import subprocess

llvm_bin = "/opt/homebrew/opt/llvm/bin"
os.environ["PATH"] = llvm_bin + os.pathsep + os.environ.get("PATH", "")
print("PATH starts with:", os.environ["PATH"].split(os.pathsep)[0])
print("\n")
# show where mlir-opt is coming from (or nothing)
print(subprocess.run(["which", "mlir-opt"], capture_output=True, text=True).stdout.strip())

# show first lines of help to confirm binary is functional
print(subprocess.run(["mlir-opt", "--help"], capture_output=True, text=True).stdout.splitlines()[:5])

PATH starts with: /opt/homebrew/opt/llvm/bin


/opt/homebrew/opt/llvm/bin/mlir-opt
['OVERVIEW: MLIR modular optimizer driver', '', 'Available Dialects: acc, affine, amdgpu, amx, arith, arm_neon, arm_sme, arm_sve, async, bufferization, builtin, cf, complex, dlti, emitc, func, gpu, index, irdl, linalg, llvm, math, memref, mesh, ml_program, mpi, nvgpu, nvvm, omp, pdl, pdl_interp, polynomial, ptr, quant, rocdl, scf, shape, sparse_tensor, spirv, tensor, tosa, transform, ub, vector, x86vector, xegpu', 'USAGE: mlir-opt [options] <input file>', '']


# Running and Testing a Lowering
### https://www.jeremykun.com/2023/08/10/mlir-running-and-testing-a-lowering/

Variable names are prefixed with %, functions by @, and each variable/value in a program has a type, often expressed after a colon. In this case all the types are i32, except for the function type which is (i32) -> i32 (not specified explicitly above, but you’ll see it in the func.call in the next code snippet).

Each statement is anchored around an expression like math.ctlz which specifies the dialect math and the operation ctlz. The rest of the syntax of the operation is determined by a parser defined by the dialect, and so many operations will have different syntaxes, though many are pulled from a fixed set of options we’ll see later in the series. In the simple case of math.ctlz, the sole argument is the integer whose leading zeros are to be counted, and the trailing : i32 denotes the output type.

It’s also important to note that func is itself a dialect, and func.func is considered an “operation,” where the braces and the function’s body is part of the syntax. In MLIR a set of operations within braces is called a region, and an operation can have zero or many regions.

There is a lot more to say about regions, and their cousins “basic blocks,” but in brief: operations may have attached regions, like the body of a for loop, and each region is a list of blocks (with an implicit block if non is explicitly listed). A block is a list of operations that is guaranteed to have only one entry and exit point. I think of the label in a block as the destination of a jump command in assembly languages. A block has exactly one “jumping in” point and one “jumping out” point. It has a more precise definition that aligns with the classical compiler concept of a basic block.

Also note, in MLIR multiple dialects often coexist in the same program as it is progressively lowered to some final backend target.

In this example of lowering we apply 6 passes:

1️⃣ convert-math-to-funcs{convert-ctlz}
Converts math operations (from the math dialect) to calls to helper functions. For example

%0 = math.ctlz %arg0 : i32
→ becomes something like a func.call to a lowered ctlz function.
Effect: dialect lowering, replaces operations that don’t have direct LLVM equivalents with function calls.

2️⃣ func.func(convert-scf-to-cf, convert-arith-to-llvm)
This is a nested pipeline for each func.func:

a) convert-scf-to-cf
Converts structured control flow (scf) into control flow (cf).
Example: loops and scf.if → cf.br, cf.cond_br (explicit branches).
Your current MLIR doesn’t have loops or scf ops, so this pass may not change @add_example.

b) convert-arith-to-llvm
Converts arith dialect operations to LLVM dialect ops.
Example:
%res = arith.addf %arg0, %c0 : f32
→ becomes:
%res = llvm.fadd %arg0, %c0 : f32
Effect: arithmetic is now expressed in LLVM dialect instead of arith.

3️⃣ convert-func-to-llvm
Lowers func.func and func.return into LLVM function equivalents.
Example:
func.func @add_example(%arg0: f32) -> f32 {
  ...
  return %res : f32
}
→ becomes:
llvm.func @add_example(%arg0: f32) -> f32 {
  ...
  llvm.return %res : f32
}
This is required before generating native code or further LLVM passes.

4️⃣ convert-cf-to-llvm
Converts control-flow dialect (cf) to LLVM dialect.
Example:
cf.br %cond, ^bb1
→
llvm.br %cond, ^bb1
Makes all branches compatible with LLVM backend.

5️⃣ reconcile-unrealized-casts
Handles unrealized casts: temporary type conversions created during lowering.
Removes or replaces them so that the IR has valid types for LLVM lowering.
Example: arith.fadd of a f32 constant may create an unrealized_conversion op → this pass removes it.

## we use the mlir-opt binary as the main entry point to parse MLIR, run a pass, and emit the output IR.

In [190]:
# Path to the MLIR file
mlir_file = "test.mlir"

# MLIR content
mlir_content = """func.func @add_example(%arg0: f32) -> f32 {
  %c0 = arith.constant 0.0 : f32
  %res = arith.addf %arg0, %c0 : f32
  return %res : f32
}
func.func @main(%arg0: i32) -> i32 {
  %0 = math.ctlz %arg0 : i32
  func.return %0 : i32
}
"""

# Write to file
with open(mlir_file, "w") as f:
    f.write(mlir_content)

# Verify it was created
import os
print("Created file:", os.path.abspath(mlir_file))
print("File contents:")
with open(mlir_file) as f:
    print(f.read())

Created file: /Users/hafsahshahzad/Desktop/MyDocs/TheCompilerLab/TheCompilerLab/MLIR/MLIR_Passes/test.mlir
File contents:
func.func @add_example(%arg0: f32) -> f32 {
  %c0 = arith.constant 0.0 : f32
  %res = arith.addf %arg0, %c0 : f32
  return %res : f32
}
func.func @main(%arg0: i32) -> i32 {
  %0 = math.ctlz %arg0 : i32
  func.return %0 : i32
}



In [191]:
mlir_file = "test.mlir"

pipeline = (
    "builtin.module(convert-math-to-funcs{convert-ctlz},"
    "func.func(convert-scf-to-cf,convert-arith-to-llvm),"
    "convert-func-to-llvm,"
    "convert-cf-to-llvm,"
    "reconcile-unrealized-casts)"
)

# Run mlir-opt and capture output --- Overall Effect on Your MLIR
#We started with high-level IR (arith, math, func) After the pipeline: LLVM-compatible IR, with all functions, 
#arithmetic, and math ops lowered.This is the standard path from MLIR → LLVM IR → native code

#@add_example:
#%res = arith.addf %arg0, %c0 → %res = llvm.fadd %arg0, %c0
#func.return → llvm.return
#Now fully lowered to LLVM dialect, ready for code generation.

#@main:
#math.ctlz → converted to a function call (func.call)
#Control flow ops converted to cf then to llvm dialect
#Unrealized casts reconciled
#Fully lowered to LLVM dialect.

result = subprocess.run(
    ["mlir-opt", mlir_file, f"--pass-pipeline={pipeline}"],
    capture_output=True,
    text=True
)

# Check for errors
if result.returncode != 0:
    print("Error running mlir-opt:")
    print(result.stderr)
else:
    print("MLIR after passes:")
    print(result.stdout)

MLIR after passes:
module {
  llvm.func @add_example(%arg0: f32) -> f32 {
    %0 = llvm.mlir.constant(0.000000e+00 : f32) : f32
    %1 = llvm.fadd %arg0, %0 : f32
    llvm.return %1 : f32
  }
  llvm.func @main(%arg0: i32) -> i32 {
    %0 = llvm.call @__mlir_math_ctlz_i32(%arg0) : (i32) -> i32
    llvm.return %0 : i32
  }
  llvm.func linkonce_odr @__mlir_math_ctlz_i32(%arg0: i32) -> i32 attributes {sym_visibility = "private"} {
    %0 = llvm.mlir.constant(32 : i32) : i32
    %1 = llvm.mlir.constant(0 : i32) : i32
    %2 = llvm.icmp "eq" %arg0, %1 : i32
    llvm.cond_br %2, ^bb1, ^bb2
  ^bb1:  // pred: ^bb0
    llvm.br ^bb10(%0 : i32)
  ^bb2:  // pred: ^bb0
    %3 = llvm.mlir.constant(1 : index) : i64
    %4 = llvm.mlir.constant(1 : i32) : i32
    %5 = llvm.mlir.constant(32 : index) : i64
    %6 = llvm.mlir.constant(0 : i32) : i32
    llvm.br ^bb3(%3, %arg0, %6 : i64, i32, i32)
  ^bb3(%7: i64, %8: i32, %9: i32):  // 2 preds: ^bb2, ^bb8
    %10 = llvm.icmp "slt" %7, %5 : i64
    llvm.cond

# Writing a Pass in MLIR
The main work in MLIR is defining passes that either optimize part of a program, lower from parts of one dialect to others, or perform various normalization and canonicalization operations. In this article, we’ll start by defining a pass that operates entirely within a given dialect by fully unrolling loops. Then we’ll define a pass that does a simple replacement of one instruction with another. 
Notes on Passes:
https://mlir.llvm.org/docs/PassManagement/

All passes in MLIR derive from OperationPass. 

________________________________________________________________________________

The pattern rewrite engine: It is useful in the kind of situation where one wants to repeatedly apply the same subset of transformations to a given IR substructure until that substructure is completely removed.A rewrite pattern is a subclass of OpRewritePattern, and it has a method called matchAndRewrite which performs the transformation.Its general syntax: 

struct AddZeroSimplify : public OpRewritePattern<arith::AddFOp> {
  using OpRewritePattern<arith::AddFOp>::OpRewritePattern;

  LogicalResult matchAndRewrite(arith::AddFOp op,
                                PatternRewriter &rewriter) const override {
                                }
};            
The return value of matchAndRewrite is a LogicalResult, which is a wrapper around a boolean to signal success or failure, along with named utility functions like failure() and success() to generate instances, and failed(...) to test for failure. LogicalResult also comes with a subclass FailureOr that is subclass of optional that inter-operates with LogicalResult via the presence or absence of a value.

________________________________________________________________________________
Then we instantiate the pattern inside the pass
To create an agnostic operation pass, a derived class must adhere to the following:

-> Inherit from the CRTP class OperationPass.
-> Override the virtual void runOnOperation() method.

// A pass that invokes the pattern rewrite engine.

void AffineFullUnrollPassAsPatternRewrite::runOnOperation() {
  mlir::RewritePatternSet patterns(&getContext());
  patterns.add<AffineFullUnrollPattern>(&getContext());
  // One could use GreedyRewriteConfig here to slightly tweak the behavior of
  // the pattern application.
  (void)applyPatternsAndFoldGreedily(getOperation(), std::move(patterns));
}
________________________________________________________________________________
Some notes:

-> Value is the type that represents an SSA value (i.e., an MLIR variable), and getDefiningOp fetches the unique operation that defines it in its scope.
-> There are a variety of “casting” operations like rhs.getDefiningOp<arith::ConstantIntOp>() that take the type you want as output as a template parameter, and return null if the type cannot be converted. You might also see cast<>, dyn_cast<>, or dyn_cast_or_null<> to invoke these manually.
-> (value & (value - 1)) is a classic bit-twiddling trick to compute if an integer is a power of two. We check it and skip the pattern if it’s not.
-> The actual constant itself is represented as an MLIR attribute, which is essentially compile-time static data attached to the op. You can put strings or dictionaries as attributes, but for ConstantOp it’s just an int.

-> The rewriter.create part is where we actually do the real work. Create a new constant that is half the original constant, create new multiplication and addition ops... 
-> rewriter.replaceOp removes the original multiplication op and uses the output of newAdd for any other operations that used the original multiplication op’s output.

 Commnd to find files: find  /opt/homebrew/opt/llvm/include/mlir -name MlirOptMain.h
all these files are /opt/homebrew/opt/llvm/include

In [192]:
%%writefile lib/SimplifyAdd.h
#ifndef SIMPLIFY_ADD_H
#define SIMPLIFY_ADD_H

#include "mlir/Pass/Pass.h"
#include "mlir/Dialect/Func/IR/FuncOps.h"

namespace mlir {
  std::unique_ptr<mlir::Pass> createSimplifyAddPass();
  void registerSimplifyAddPass();
}
#endif 


Overwriting lib/SimplifyAdd.h


In [193]:
%%writefile lib/SimplifyAdd.cpp
#include "SimplifyAdd.h"         // your new header (adjust path if needed)

#include "mlir/IR/PatternMatch.h"
#include "mlir/Pass/Pass.h"
#include "mlir/Dialect/Func/IR/FuncOps.h"
#include "mlir/Dialect/Arith/IR/Arith.h"
#include "mlir/Transforms/GreedyPatternRewriteDriver.h"
#include "mlir/IR/Dialect.h"
using namespace mlir;
// Pattern: arith.addf %x, 0.0 -> %x
struct AddZeroSimplify : public OpRewritePattern<arith::AddFOp> {
  using OpRewritePattern<arith::AddFOp>::OpRewritePattern;

  LogicalResult matchAndRewrite(arith::AddFOp op,
                                PatternRewriter &rewriter) const override {
    auto lhs = op.getOperand(0);

    // canonicalization patterns ensure the constant is on the right, if there is a constant
    // See https://mlir.llvm.org/docs/Canonicalization/#globally-applied-rules
    auto rhs = op.getOperand(1);
    auto rhsDefiningOp = rhs.getDefiningOp<arith::ConstantOp>();
    if (!rhsDefiningOp) {
      return failure();
    }
    auto attr = mlir::dyn_cast<FloatAttr>(rhsDefiningOp.getValue());
    if (!attr) return failure();
    if (attr.getValueAsDouble() == 0.0) {
      rewriter.replaceOp(op, lhs);
      return success();
    }
    return failure();

  }
};

//All passes in MLIR derive from OperationPass 
struct SimplifyAddPass
    : public PassWrapper<SimplifyAddPass, OperationPass<ModuleOp>> {
  
  void runOnOperation() override {
    RewritePatternSet patterns(&getContext());
    patterns.add<AddZeroSimplify>(&getContext());
    (void)applyPatternsGreedily(getOperation(), std::move(patterns));
  }
  StringRef getArgument() const final { return "simplify-add"; }
  StringRef getDescription() const final { return "Simplify x + 0.0 -> x"; }

};
// Legacy registration — works with Homebrew MLIR
namespace {
  PassRegistration<SimplifyAddPass> pass;
}

namespace mlir {
  void registerSimplifyAddPass() {
    PassRegistration<SimplifyAddPass>();
  }
}


// Factory function definition
std::unique_ptr<mlir::Pass> mlir::createSimplifyAddPass() {
  return std::make_unique<SimplifyAddPass>();
}



Overwriting lib/SimplifyAdd.cpp


In [194]:
%%writefile tools/mlir-opt-custom.cpp
#include "mlir/IR/MLIRContext.h"
#include "mlir/IR/Dialect.h"
#include "mlir/Pass/PassManager.h"
#include "mlir/Pass/PassRegistry.h"
#include "mlir/Parser/Parser.h"
#include "mlir/InitAllDialects.h"
#include "mlir/Tools/mlir-opt/MlirOptMain.h"
#include "mlir/Conversion/Passes.h"
#include "mlir/Conversion/FuncToLLVM/ConvertFuncToLLVMPass.h"
#include "mlir/Conversion/ControlFlowToLLVM/ControlFlowToLLVM.h"
#include "mlir/Dialect/LLVMIR/LLVMDialect.h"

// Include your pass header
#include "../lib/SimplifyAdd.h"

using namespace mlir;

int main(int argc, char **argv) {
  // Create a registry and register all MLIR dialects
  DialectRegistry registry;
  registry.insert<mlir::arith::ArithDialect,
                mlir::func::FuncDialect,
                mlir::math::MathDialect,
                LLVM::LLVMDialect>();
  mlir::registerConvertFuncToLLVMPass();
  mlir::registerConvertControlFlowToLLVMPass();

  // Configure mlir-opt to run your pass
  MlirOptMainConfig config;
  config.setPassPipelineSetupFn([](PassManager &pm) -> LogicalResult {
      pm.addPass(createSimplifyAddPass());
      return success();
  });

  // Run mlir-opt main with CLI arguments
  return asMainReturnCode(MlirOptMain(argc, argv, "simplify-add-opt", registry));
}

Overwriting tools/mlir-opt-custom.cpp


In [195]:
%%writefile CMakeLists.txt
cmake_minimum_required(VERSION 3.20)
project(MLIRPasses LANGUAGES C CXX)

# Homebrew LLVM/MLIR paths
set(LLVM_DIR "/opt/homebrew/opt/llvm/lib/cmake/llvm")
set(MLIR_DIR "/opt/homebrew/opt/llvm/lib/cmake/mlir")

# Find MLIR
find_package(MLIR REQUIRED CONFIG)

# Include MLIR headers and definitions
add_definitions(${MLIR_DEFINITIONS})
include_directories(${MLIR_INCLUDE_DIRS})

# Source files for your executable
set(SOURCES
    tools/mlir-opt-custom.cpp
    lib/SimplifyAdd.cpp
)

# Create executable
add_executable(mlir-opt-custom ${SOURCES})

# Link against MLIR libraries (modern replacements)
target_link_libraries(mlir-opt-custom
    PRIVATE
    MLIRIR
    MLIRPass
    MLIRSupport
    MLIRTransforms
    MLIRParser
    MLIRRewrite
    MLIRAnalysis
    MLIRFuncDialect      # needed for func::FuncOp
    MLIRArithDialect     # needed for arith::AddFOp\
    MLIRMathDialect
    MLIROptLib
    MLIRLLVMDialect
    MLIRFuncToLLVM
    MLIRControlFlowToLLVM
)

# Set C++ standard
set_target_properties(mlir-opt-custom PROPERTIES
    CXX_STANDARD 17
    CXX_STANDARD_REQUIRED YES
    CXX_EXTENSIONS NO
)


Overwriting CMakeLists.txt


In [196]:
%%bash
cd /Users/hafsahshahzad/Desktop/MyDocs/TheCompilerLab/TheCompilerLab/MLIR/MLIR_Passes
rm -rf build
mkdir build
cd build

cmake -G Ninja -DCMAKE_OSX_ARCHITECTURES=arm64 ..
ninja


-- The C compiler identification is Clang 20.1.8
-- The CXX compiler identification is Clang 20.1.8
-- Detecting C compiler ABI info
-- Detecting C compiler ABI info - done
-- Check for working C compiler: /opt/homebrew/opt/llvm/bin/clang - skipped
-- Detecting C compile features
-- Detecting C compile features - done
-- Detecting CXX compiler ABI info
-- Detecting CXX compiler ABI info - done
-- Check for working CXX compiler: /opt/homebrew/opt/llvm/bin/clang++ - skipped
-- Detecting CXX compile features
-- Detecting CXX compile features - done
-- Performing Test HAVE_FFI_CALL
-- Performing Test HAVE_FFI_CALL - Success
-- Found FFI: /Library/Developer/CommandLineTools/SDKs/MacOSX.sdk/usr/lib/libffi.tbd
-- Looking for histedit.h
-- Looking for histedit.h - found
-- Found LibEdit: /Library/Developer/CommandLineTools/SDKs/MacOSX.sdk/usr/include (found version "2.11")
-- Found ZLIB: /Library/Developer/CommandLineTools/SDKs/MacOSX.sdk/usr/lib/libz.tbd (found version "1.2.12")
-- Found zstd

In [197]:
#!mlir-opt -load ./build/libSimplifyAdd.dylib \
#    --pass-pipeline="simplify-add,convert-func-to-llvm,convert-cf-to-llvm" \
#    test.mlir

!./build/mlir-opt-custom \
    --pass-pipeline="builtin.module(simplify-add,convert-func-to-llvm,convert-cf-to-llvm)" \
    test.mlir


module {
  llvm.func @add_example(%arg0: f32) -> f32 {
    llvm.return %arg0 : f32
  }
  llvm.func @main(%arg0: i32) -> i32 {
    %0 = math.ctlz %arg0 : i32
    llvm.return %0 : i32
  }
}



In [198]:
%%writefile test_pass1.mlir
module {
  // x + 0.0 → should be simplified
  func.func @add_zero(%arg0: f32) -> f32 {
    %c0 = arith.constant 0.0 : f32
    %0 = arith.addf %arg0, %c0 : f32
    return %0 : f32
  }

  // x + 1.0 → should remain unchanged
  func.func @add_nonzero(%arg0: f32) -> f32 {
    %c1 = arith.constant 1.0 : f32
    %0 = arith.addf %arg0, %c1 : f32
    return %0 : f32
  }

  // Nested addition: (x + 0.0) + 0.0 → inner and outer should simplify
  func.func @nested_add(%arg0: f32) -> f32 {
    %c0 = arith.constant 0.0 : f32
    %0 = arith.addf %arg0, %c0 : f32
    %1 = arith.addf %0, %c0 : f32
    return %1 : f32
  }

  // Mixed: x + 0.0 + 1.0 → only the +0.0 should simplify
  func.func @mixed_add(%arg0: f32) -> f32 {
    %c0 = arith.constant 0.0 : f32
    %c1 = arith.constant 1.0 : f32
    %0 = arith.addf %arg0, %c0 : f32
    %1 = arith.addf %0, %c1 : f32
    return %1 : f32
  }
}


Overwriting test_pass1.mlir


In [199]:
!./build/mlir-opt-custom \
    --pass-pipeline="builtin.module(simplify-add,convert-func-to-llvm,convert-cf-to-llvm)" \
    test_pass1.mlir


module {
  llvm.func @add_zero(%arg0: f32) -> f32 {
    llvm.return %arg0 : f32
  }
  llvm.func @add_nonzero(%arg0: f32) -> f32 {
    %cst = arith.constant 1.000000e+00 : f32
    %0 = arith.addf %arg0, %cst : f32
    llvm.return %0 : f32
  }
  llvm.func @nested_add(%arg0: f32) -> f32 {
    llvm.return %arg0 : f32
  }
  llvm.func @mixed_add(%arg0: f32) -> f32 {
    %cst = arith.constant 1.000000e+00 : f32
    %0 = arith.addf %arg0, %cst : f32
    llvm.return %0 : f32
  }
}



### Observations:

1. All + 0.0 operations are removed.
2. + 1.0 stays untouched.
3. Nested simplifications are applied recursively.

-G Ninja → uses the Ninja build system (already installed via Homebrew).
-DMLIR_DIR=... → points CMake to your MLIR installation.
.. → tells CMake to use your project root (where CMakeLists.txt is) as the source.

In [200]:
!which mlir-opt
!otool -L build/lib/libSimplifyAdd.dylib

/opt/homebrew/opt/llvm/bin/mlir-opt
error: /Library/Developer/CommandLineTools/usr/bin/otool-classic: can't open file: build/lib/libSimplifyAdd.dylib (No such file or directory)
